# 🏗️ Notebook 1: Reddit — Requirements & Architecture


## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Describe the **functional** and **non-functional** requirements of a Reddit-like system.
- Do a **back-of-envelope** estimate (QPS, storage, bandwidth) for 50 M daily users.
- Explain why Reddit is a **read-heavy**, **eventually consistent** system.
- Sketch a high-level architecture and justify each service boundary.
- See, with a tiny simulation, **why we must precompute the Hot feed** instead of computing it per request.


## 🛠️ Setup

```bash
cd 06-system-designs/reddit
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. What are we actually building?

Reddit is a **social news aggregator**. Users gather into communities called *subreddits*
(`r/python`, `r/cats`, ...), post links or text, **upvote** or **downvote** others' posts,
and reply to each other in **threaded** comments. The platform ranks content so that "what's
interesting right now" bubbles to the top.

The four screens you must support:

| Screen | Reads from | Writes to |
|---|---|---|
| Subreddit feed (`/r/python?sort=hot`) | precomputed ranked feed | — |
| Post detail + comments | post row + comment tree | — |
| Vote button | — | votes table / stream |
| Submit post / comment | — | posts / comments table |

### Functional requirements

1. Create subreddits; users can subscribe/unsubscribe.
2. Submit posts (link or text). Edit / delete own posts.
3. Upvote / downvote / unvote on posts **and** comments.
4. Threaded comments (a reply to a reply to a reply …).
5. Feeds with multiple sort orders: **Hot**, **New**, **Top**, **Rising**, **Controversial**.
6. A cross-subreddit `r/all` feed.

### Non-functional requirements

- **Read-heavy**: ~99 % of traffic is reads (browsing feeds, reading comments). Writes are votes,
  posts, and comments.
- **Eventually consistent scores are fine**: no one notices if an upvote takes a few seconds to
  appear on the counter.
- **Hot ranking is time-sensitive**: the Hot feed must change every few minutes, so some piece
  of the system must re-rank continuously.
- **Comment trees can be huge**: viral threads have 100 k+ comments. We cannot load the whole
  tree; we must paginate within a branch ("load more replies").
- **Voting must be cheap and racy-safe**: millions of votes per day on a few mega-posts.


## 2. Back-of-envelope numbers

We can't design for scale we haven't estimated. Let's ballpark it.


In [ ]:
# A tiny back-of-envelope calculator.
# Numbers are illustrative — the *method* is what matters in an interview.
# Rule: every number quoted in the prose below must come OUT of this cell.

DAU = 50_000_000           # daily active users
PAGE_LOADS_PER_USER = 5    # feed/post views per user per day (conservative)
VOTES_PER_USER = 0.2       # 1 vote every 5 days on average
POSTS_PER_USER = 0.01      # 1 post per 100 users per day
COMMENTS_PER_USER = 0.1    # 1 comment per 10 users per day

SECONDS_PER_DAY = 86_400
PEAK_FACTOR = 10           # traffic is not flat: peak hour ~10x the daily mean

read_rps  = DAU * PAGE_LOADS_PER_USER / SECONDS_PER_DAY
vote_rps  = DAU * VOTES_PER_USER      / SECONDS_PER_DAY
post_rps  = DAU * POSTS_PER_USER      / SECONDS_PER_DAY
comm_rps  = DAU * COMMENTS_PER_USER   / SECONDS_PER_DAY
write_rps = vote_rps + post_rps + comm_rps

print("QPS")
print(f"  feed reads : {read_rps:>10,.0f} rps   ({read_rps*PEAK_FACTOR:>10,.0f} at {PEAK_FACTOR}x peak)")
print(f"  votes      : {vote_rps:>10,.0f} rps   ({vote_rps*PEAK_FACTOR:>10,.0f} at {PEAK_FACTOR}x peak)")
print(f"  new posts  : {post_rps:>10,.0f} rps")
print(f"  comments   : {comm_rps:>10,.0f} rps")
print(f"  read:write ratio = {read_rps/write_rps:,.0f} : 1")


Note how modest that ratio is. **5 page loads/user/day is a conservative reader.** Reddit's
real read:write ratio is far higher because a scrolling session is dozens of feed pages plus a
comment page each. Re-run the cell with `PAGE_LOADS_PER_USER = 50` and the ratio jumps to
~280:1. The *shape* of the conclusion ("overwhelmingly read-heavy") is robust to the input;
the exact multiplier is not. Say that out loud in an interview.

### Storage and bandwidth — derived from the same constants


In [ ]:
# Storage and bandwidth must fall out of the SAME assumptions as the QPS above,
# otherwise the estimate quietly contradicts itself.

POST_BYTES    = 1_000     # title + body + metadata
COMMENT_BYTES = 400       # comments are short but there are a LOT of them
VOTE_BYTES    = 24        # (user_id, target_id, value) + row overhead
FEED_PAGE_KB  = 60        # JSON for 25 posts rendered into a feed page

posts_per_day    = DAU * POSTS_PER_USER
comments_per_day = DAU * COMMENTS_PER_USER
votes_per_day    = DAU * VOTES_PER_USER

print("Daily volume")
print(f"  posts    : {posts_per_day:>15,.0f}")
print(f"  comments : {comments_per_day:>15,.0f}")
print(f"  votes    : {votes_per_day:>15,.0f}")

TB = 1e12
for years in (1, 10):
    days = 365 * years
    post_tb = posts_per_day    * days * POST_BYTES    / TB
    comm_tb = comments_per_day * days * COMMENT_BYTES / TB
    # Votes are the one table we do NOT keep forever: we store only the LATEST
    # vote per (user, target), so the row count is bounded by what users have
    # actually voted on, not by the historical vote stream.
    vote_tb = votes_per_day    * days * VOTE_BYTES    / TB
    print(f"\nAfter {years:>2} year(s):")
    print(f"  posts    : {post_tb:>8.2f} TB   ({posts_per_day*days:>15,.0f} rows)")
    print(f"  comments : {comm_tb:>8.2f} TB   ({comments_per_day*days:>15,.0f} rows)")
    print(f"  votes    : {vote_tb:>8.2f} TB   ({votes_per_day*days:>15,.0f} rows, before dedup)")
    print(f"  TOTAL    : {post_tb+comm_tb+vote_tb:>8.2f} TB (before replication/indexes)")

# Egress: feeds dominate, because every read ships a page of JSON.
egress_gbps      = read_rps * FEED_PAGE_KB * 1024 * 8 / 1e9
peak_egress_gbps = egress_gbps * PEAK_FACTOR
print(f"\nFeed egress : {egress_gbps:.1f} Gbps average, {peak_egress_gbps:.1f} Gbps at peak")
print("  -> comfortably CDN-cacheable; the text payload is small next to images/video,")
print("     which we offload to object storage + CDN and never serve from the app tier.")


**Takeaway:** with these inputs the system is **read-dominated** (see the printed ratio).
That single fact drives *every* design decision:

- Cache everything readable.
- Use eventual consistency for counters (it's fine).
- Put votes on a log/stream and batch them; don't synchronously update a row per vote.

### What the storage numbers tell us

- **Posts are tiny.** Even after 10 years the post table is a couple of TB — one well-indexed
  sharded MySQL cluster handles it.
- **Comments are ~10x posts** by row count. This is the table that actually needs sharding
  (by `post_id`, so one post's whole thread stays co-located).
- **Votes are the biggest table by far**, which is exactly why we keep only the *latest* vote
  per `(user, target)` rather than an append-only vote history. The denormalised `ups`/`downs`
  counters on the post row are what feeds actually read; the `votes` table exists for
  "did I vote on this?" and for recounting after drift.


## 3. High-level architecture

```
               ┌──────────┐
               │  client  │
               └────┬─────┘
                    ▼
              ┌──────────┐
              │ API gtwy │   auth, rate limit, routing
              └────┬─────┘
         ┌────────┬┴────────┬──────────┬─────────────┐
         ▼        ▼         ▼          ▼             ▼
        Feed    Post      Vote       Comment      Subreddit
        Svc     Svc       Svc         Svc            Svc
         │       │         │            │             │
         ▼       ▼         ▼            ▼             ▼
       Redis   MySQL     Kafka        MySQL         MySQL
       (hot    (canonical (votes →   (threaded
        feed)  posts)     ranking)    comments)
                            │
                            ▼
                  ┌──────────────────────┐
                  │  Ranking worker      │  runs every ~5 min
                  │  reads votes + age   │  → writes hot_score
                  │  → updates Redis feed│
                  └──────────────────────┘
```

### Why these boundaries?

- **Feed Svc**: its job is to serve *pre-ranked* lists. It never runs the ranking formula on
  the hot path; it reads from Redis.
- **Vote Svc**: accepts votes, writes to Kafka, returns 200 immediately. The heavy lifting (updating
  counters, re-ranking) happens downstream. This decouples the user-facing latency from the
  aggregation work.
- **Ranking worker**: consumes votes, recomputes `hot_score`, writes to the feed cache.
  It *is* allowed to be a few minutes stale — users expect that.
- **Post / Comment / Subreddit Svc**: classic CRUD services backed by MySQL.

The MySQL schema is shown in Notebook 2; the algorithms (hot ranking, sharded counters,
comment trees) are in Notebook 3.


## 4. Simulation — why the Hot feed must be precomputed

Imagine we have 1 million posts and a request wants the **top 25 by hot score**. Two strategies:

- **Compute-on-read** — score every post at request time, sort, return top 25.
- **Precomputed** — a background job already wrote the sorted list to Redis; we just `LRANGE`.

Let's time them.


In [ ]:
import math, random, time, heapq

random.seed(42)
NOW = int(time.time())

# simulate 1,000,000 posts with random creation time + votes
posts = [
    (pid,
     random.randint(0, 50_000),                  # ups
     random.randint(0, 5_000),                   # downs
     NOW - random.randint(0, 7 * 86_400))        # created_at (unix seconds, last 7 days)
    for pid in range(1_000_000)
]

def hot_score(ups, downs, created_at):
    """Reddit's hot ranking. Note the score rises with CREATION TIME, not with age —
    getting that backwards ranks the oldest posts highest. Full derivation in notebook 3."""
    net = ups - downs
    sign = 1 if net > 0 else (-1 if net < 0 else 0)
    order = math.log10(max(abs(net), 1))
    return sign * order + created_at / 45_000

# --- precomputed: a background job already produced the sorted list.
#     We build it OUTSIDE the timer, exactly as the ranking worker would.
ranked_ids = [p[0] for p in heapq.nlargest(25, posts, key=lambda p: hot_score(p[1], p[2], p[3]))]

# --- compute-on-read: score all 1M posts on the request path
t0 = time.perf_counter()
top25_live = heapq.nlargest(25, posts, key=lambda p: hot_score(p[1], p[2], p[3]))
live_ms = (time.perf_counter() - t0) * 1000

# --- precomputed read: just slice the cached list (an LRANGE against Redis)
t0 = time.perf_counter()
top25_cached = ranked_ids[:25]
cached_ms = (time.perf_counter() - t0) * 1000

assert [p[0] for p in top25_live] == top25_cached, "cached list must match the live ranking"
print(f"compute-on-read : {live_ms:9.3f} ms   (scored 1,000,000 posts)")
print(f"precomputed read: {cached_ms:9.3f} ms   (read 25 ids from the cache)")
print(f"speedup         : {live_ms/max(cached_ms, 1e-6):,.0f}x")
print("\nThe precomputed cost is O(page size). The live cost is O(total posts) —")
print("it gets worse every single day the site stays up.")


The precomputed version is **thousands of times faster** and, crucially, its cost does **not**
grow with the total number of posts. At 30 k requests/second you *must* precompute.

The price we pay is **freshness**: the feed is always a few minutes stale. For a news-aggregator
that's an acceptable trade; for a stock ticker it would not be.


- Reddit is **overwhelmingly read-heavy** (compute the exact ratio; don't memorise one) → optimise
  for reads, tolerate eventual consistency on the write side.